In [ ]:

task_file = "/home/yu/XRT/MultipleTiling/cond/inject/tasks_inject.txt"
tasks = []
with open(task_file) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        icdir, idx = line.split()
        tasks.append((icdir, int(idx)))

len(tasks), tasks[:3]

(35000,
 [('/home/yu/XRT/MultipleTiling/IC160731A', 0),
  ('/home/yu/XRT/MultipleTiling/IC160731A', 1),
  ('/home/yu/XRT/MultipleTiling/IC160731A', 2)])

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import pickle
from pathlib import Path
from tqdm import tqdm
import numpy as np
from inject_mp_rho import run
LZGamma = 7  ## changeable
nproc = 25
ctx = mp.get_context("fork")

# rho_list = np.array([2e-6, 2e-7, 4e-7, 6e-7, 8e-7, 2e-8, 4e-8, 6e-8, 8e-8, 2e-9, 4e-9, 6e-9, 8e-9, 2e-10, 4e-10, 6e-10, 8e-10])
# rho_list = np.array([1e-10, 3e-10, 6e-10, 1e-9, 3e-9, 6e-9, 1e-8, 3e-8, 6e-8, 1e-7, 3e-7, 6e-7, 1e-6, 3e-6])
rho_list = np.array([1e-10, 3e-10, 6e-10, 1e-9, 3e-9, 6e-9, 1e-8, 3e-8, 6e-8, 1e-7, 3e-7, 6e-7, 1e-6, 3e-6])
# rho_list = np.array([6e-9, 3e-9, 1e-8, 3e-8, 6e-8, 1e-7, 3e-7, 6e-7, 1e-6, 3e-6])
# rho_list = np.array([4e-10, 6e-10, 8e-10])
# rho_list = np.array([4e-6])

save_root = Path(f"/home/yu/XRT/MultipleTiling/summary_gamma{LZGamma:.0f}/")
save_root.mkdir(parents=True, exist_ok=True)

for rho in rho_list:
    rho = float(rho)
    ok, ng = 0, 0
    failures = []
    results = []

    with ProcessPoolExecutor(max_workers=nproc, mp_context=ctx) as ex:
        futs = {ex.submit(run, icdir, idx, rho): (icdir, idx) for (icdir, idx) in tasks}

        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"rho={rho:.1e}"):
            icdir, idx = futs[fut]
            try:
                res = fut.result()          # res は dict
                results.append(res)
                if res.get("status") == "ok":
                    ok += 1
                else:
                    ng += 1
                    failures.append(((icdir, idx), res.get("status"), res.get("note")))
            except Exception as e:
                ng += 1
                failures.append(((icdir, idx), "future_exception", repr(e)))
                print("[FAILED]", (icdir, idx), e)

    print(f"Done rho={rho:.1e}. OK={ok} NG={ng}")

    # ---- 保存（rhoごと）----

    tag = f"rho{rho:.1e}".replace("+", "")
    out_pkl = save_root / f"results_{tag}.pkl"
    out_fail = save_root / f"failures_{tag}.pkl"

    with out_pkl.open("wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

    with out_fail.open("wb") as f:
        pickle.dump(failures, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("saved:", out_pkl, out_fail)

event selection type = gfu
Ntot (atmos numu) = 1.49e+05 [1/yr]
Ntot (signal numu) = 6.87e+01 [1/yr]
Ntot (residual numu) = 2.89e+02 [1/yr]


/misc/home/yu/XRT/MultipleTiling/cond/inject_unified/inject_mp_rho.py:196: RuntimeWarning: invalid value encountered in divide
  signalness = sig_tot*pdf_sig/(sig_tot*pdf_sig+dif_tot*pdf_dif+atmos_tot*pdf_atmos)
rho=1.0e-10: 100%|██████████| 35000/35000 [1:01:57<00:00,  9.42it/s]


Done rho=1.0e-10. OK=31000 NG=4000
saved: /home/yu/XRT/MultipleTiling/summary_gamma7/results_rho1.0e-10.pkl /home/yu/XRT/MultipleTiling/summary_gamma7/failures_rho1.0e-10.pkl


rho=3.0e-10: 100%|██████████| 35000/35000 [56:52<00:00, 10.26it/s]   


Done rho=3.0e-10. OK=31000 NG=4000
saved: /home/yu/XRT/MultipleTiling/summary_gamma7/results_rho3.0e-10.pkl /home/yu/XRT/MultipleTiling/summary_gamma7/failures_rho3.0e-10.pkl


...

Done rho=6.0e-07. OK=31000 NG=4000
saved: /home/yu/XRT/MultipleTiling/summary_gamma7/results_rho6.0e-07.pkl /home/yu/XRT/MultipleTiling/summary_gamma7/failures_rho6.0e-07.pkl


rho=1.0e-06: 100%|██████████| 35000/35000 [1:22:06<00:00,  7.10it/s]  


Done rho=1.0e-06. OK=31000 NG=4000
saved: /home/yu/XRT/MultipleTiling/summary_gamma7/results_rho1.0e-06.pkl /home/yu/XRT/MultipleTiling/summary_gamma7/failures_rho1.0e-06.pkl


rho=3.0e-06: 100%|██████████| 35000/35000 [1:22:06<00:00,  7.10it/s]  


Done rho=3.0e-06. OK=31000 NG=4000
saved: /home/yu/XRT/MultipleTiling/summary_gamma7/results_rho3.0e-06.pkl /home/yu/XRT/MultipleTiling/summary_gamma7/failures_rho3.0e-06.pkl
